# 5.1 ***层和块***

In [3]:
import torch
from torch import nn
from torch.nn import functional as F

### 5.1.1 *从零实现一个块*

块的基本功能：  
1、将输入数据作为其前向传播函数的参数。  
2、通过前向传播函数来生成输出。  
3、计算其输出关于输入的梯度，可通过其反向传播函数进行访问。通常这是自动发生的。  
4、存储和访问前向传播计算所需的参数。  
5、根据需要初始化模型参数。

In [6]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [8]:
net = MLP()
X = torch.rand(2, 20)
net(X)

tensor([[ 0.1327, -0.0174,  0.0346, -0.1016, -0.2040, -0.0009,  0.1493, -0.0973,
          0.1289,  0.0955],
        [ 0.0118, -0.0660, -0.0654,  0.0200, -0.1716, -0.0086,  0.2004, -0.0432,
          0.1769,  0.1338]], grad_fn=<AddmmBackward0>)

### 5.1.2 *顺序块*

Sequential类至少需要实现：  
1、一种将块逐个追加到列表中的函数。  
2、一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

In [11]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [12]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.0231, -0.0602,  0.2523, -0.0452, -0.4614,  0.2326, -0.2380,  0.0432,
          0.1095, -0.1634],
        [-0.0455, -0.1284,  0.2733,  0.0494, -0.3518,  0.1643, -0.0918, -0.0178,
          0.0697, -0.0115]], grad_fn=<AddmmBackward0>)

# 5.2 ***参数管理***

In [13]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[0.4638],
        [0.3589]], grad_fn=<AddmmBackward0>)

### 5.2.1 *参数访问*

从嵌套块访问参数

In [17]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[0.4062],
        [0.4062]], grad_fn=<AddmmBackward0>)

In [18]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


### 5.2.2 *参数初始化*

自定义初始化